In [41]:
import os
import sys
import time
import pickle
import warnings
from datetime import datetime, timedelta
from pathlib import Path
from difflib import SequenceMatcher
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import NearestNeighbors
import numpy as np
from scipy.optimize import minimize
from scipy.stats import dirichlet
from scipy import stats
from scipy.spatial.distance import mahalanobis
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.ticker import MultipleLocator
import seaborn as sns
from statsmodels.distributions.empirical_distribution import ECDF
from pulp import LpMaximize, LpProblem, LpVariable, lpSum
import gurobipy as gp
from scipy.interpolate import interp1d
from gurobipy import Model, GRB
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from IPython.display import display
from preproces_prod3 import *

warnings.filterwarnings("ignore")

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

path_nirsecl = path_actual.parent.parent/'Nirse_cl' / 'Data'
df_historic = pd.read_csv(path_nirsecl/"data_sintrib.csv").drop(columns=['Unnamed: 0']) 

In [42]:
df_nac_tomerge = (pd.read_csv(path_data/'df_nac_tomerge_to_simulate.csv')
                  .query('(year_nac == 2023 & month_nac <= 9) | (year_nac == 2022 & month_nac >=1)')
                  .assign(fecha_nac = lambda x: pd.to_datetime(x.fecha_nac, infer_datetime_format=True),))

df = (pd.read_csv(path_data/'df_historic_process_to_simulate.csv')
      .query('(year_nac == 2023 & month_nac <= 9) | (year_nac == 2022 & month_nac >=1)')
      .assign(fecha_nac = lambda x: pd.to_datetime(x.FECHA_NAC, infer_datetime_format=True),
              fechaIng = lambda x: pd.to_datetime(x.fechaIng, infer_datetime_format=True),
              fechaEgr = lambda x: pd.to_datetime(x.fechaEgr, format='%Y-%m-%d'),
              to_orden = lambda x: (x.fechaEgr - x.fechaIng).dt.days,)
      .query('days_mb>0 | days_upc>0')
      #.sort_values('to_orden')
     # .drop_duplicates(subset=['RUN','fecha_nac'])
)

pf_nac = pd.read_csv(path_data / "nacs_needed.csv")

def estado_salida_nirse_funct(estado_actual, eff_MA=0.764, eff_HA=0.77, eff_ICU=0.85):
    estado_salida = None
    
    if np.random.uniform()<=eff_MA:
        estado_salida='libre'
    
    elif np.random.uniform(eff_MA,1)<=eff_HA:
        estado_salida='MA'
    
    elif (estado_actual=='HA') | (np.random.uniform(eff_HA,1)<=eff_ICU):
        estado_salida='HA'
    
    elif (estado_actual=='ICU') & (np.random.uniform(eff_ICU,1)<=1): #preguntar por ICU es redundante en realidad
        estado_salida='ICU'

    return estado_salida

def cost_eff_strategy_optimize(
    df,
    fecha_catchup=pd.Timestamp("2022-10-01"),
    temporada_inicio=pd.Timestamp("2023-03-01"),
    duracion_nirse_meses=6,
    df_nac_tomerge=df_nac_tomerge,
    efectividad_nirse_ha = 0.768,
    efectividad_nirse_icu =  0.85,
    efectividad_nirse_ma = 0.764,
    reduccion_estada_nirse = 0.8
):

    efectividad_nirse_alta = efectividad_nirse_ha #0.77
    efectividad_nirse_alta_upc =  efectividad_nirse_icu #0.786 debería estarn en 0.85 si digo lode 80% estadia, para ser consistente
    cobertura_nirse =   {'newborn': 0.971, 'catchup': 0.912} # {'newborn': 1.0, 'catchup': 1.0} #

    costo_nirse_total = 297_500 /980  # precio_leo_CLP/ dolar_price =303,578 usd    ##     275   226.5 #+ 22.27
    costo_hosp_per_day = 414 
    costo_upc_per_day = 1083
    costo_emerg = 192
    ano_tem = temporada_inicio.year
    temporada_fin = pd.Timestamp(f"{ano_tem}-09-30")
    
    df = df.query('((@fecha_catchup <= fecha_nac <= @temporada_fin) & (@temporada_inicio <= fechaIng <= @temporada_fin))').copy() 
    df_nac_tomerge = df_nac_tomerge.query('@fecha_catchup <= fecha_nac <= @temporada_fin').copy()

    catchup = df[(df['fecha_nac'] >= fecha_catchup) & (df['fecha_nac'] < temporada_inicio)].copy() # El equipo Nirse
    newborn = df[(df['fecha_nac'] >= temporada_inicio)].copy() #EN TEORÍA NO SÉ CUANDO NACEN, PERO SI QUE PARA FECHA ABRYSVO AUN NO.
    
    ################################## CATCHUP #############################
    catchup['rand_cobertura'] = np.random.uniform(0, 1, size=len(catchup))
    catchup['fecha_colocacion_nirse'] = temporada_inicio + pd.to_timedelta(np.random.uniform(0, 60, size=len(catchup)), unit='d')
    catchup.loc[catchup.rand_cobertura>cobertura_nirse['catchup'], 'fecha_colocacion_nirse'] = pd.NaT
    catchup.loc[catchup.fecha_colocacion_nirse.notna(),'nirse'] = 1
    
    catchup['paso_net_inmune'] = ((catchup['fecha_colocacion_nirse'] + pd.Timedelta(days=7)) < catchup['fechaIng']).astype(int)

    noimm = catchup[catchup.paso_net_inmune == 0]
    imm   = catchup[catchup.paso_net_inmune == 1]

    count_no_time_incubar_nirse = noimm.query('nirse==1').RUN.nunique() #noimm.RUN.nunique() # ESTE CONTEO EN REALIDAD NO ES EL MEJOR, PUES DEPENDE DE COBERTURAS
    count_total = catchup.nirse.sum() # catchup.RUN.nunique()               # LO MISMO CON ESTE

    imm['meses_desde_colocacion'] = (
        (imm['fechaIng'] - imm['fecha_colocacion_nirse'])
        / pd.Timedelta(days=30)
    )
    imm['efectividad_actual'] = np.where(
        imm['meses_desde_colocacion'] <= duracion_nirse_meses,
        efectividad_nirse_alta,
        efectividad_nirse_alta / 2
    )
    imm['efectividad_actual_upc'] = np.where(
        imm['meses_desde_colocacion'] <= duracion_nirse_meses,
        efectividad_nirse_alta_upc,
        efectividad_nirse_alta_upc / 2
    )
    imm['efectividad_actual_ma'] = np.where(
        imm['meses_desde_colocacion'] <= duracion_nirse_meses,
        efectividad_nirse_ma,
        efectividad_nirse_ma / 2
    )
    
    imm['estado_actual'] = np.where(
        imm['days_upc'] > 0, 'ICU', 'HA'
    )
    
    imm['estado_salida_nirse'] = imm.apply(
        lambda r: estado_salida_nirse_funct(
            estado_actual=r['estado_actual'],
            eff_MA=r['efectividad_actual_ma'],
            eff_HA=r['efectividad_actual'],
            eff_ICU=r['efectividad_actual_upc']
        ),
        axis=1
    )
    
    conds = [
        imm['estado_salida_nirse'] == 'libre',
        imm['estado_salida_nirse'] == 'MA',
        imm['estado_salida_nirse'] == 'HA',
        imm['estado_salida_nirse'] == 'ICU'
    ]
    
    imm['days_mb_plus_upc_mapped'] = ((imm['days_mb']+imm['days_upc'])*reduccion_estada_nirse).astype(int)
    imm['days_mb_only_mapped'] = (imm['days_mb']*reduccion_estada_nirse).astype(int)
    imm['days_upc_only_mapped'] = (imm['days_upc']*reduccion_estada_nirse).astype(int)
    
    choices_mb = [0,0,imm['days_mb_plus_upc_mapped'],imm['days_mb_only_mapped']]
    choices_upc = [0,0,0,imm['days_upc_only_mapped']]

    imm['days_mb_simuled'] = np.select(conds, choices_mb, default=imm['days_mb'])
    imm['days_upc_simuled'] = np.select(conds, choices_upc, default=imm['days_upc'])
    imm['visita_urgencias'] = (imm['estado_salida_nirse'] == 'MA').astype(int)

    imm.drop(columns=['days_mb_plus_upc_mapped','days_mb_only_mapped','days_upc_only_mapped'],inplace=True) 

    catchup = pd.concat([imm, noimm], ignore_index=True)
    
    
    ################################## NEWBORN ##############################
        
    newborn['rand_cobertura_nirse'] = np.random.uniform(0, 1, size=len(newborn))
    newborn['fecha_colocacion_nirse'] = np.where(newborn['fecha_nac'] < temporada_inicio,
                                                      temporada_inicio + pd.to_timedelta(np.random.uniform(0, 60, size=len(newborn)), unit='d'), #son catchup, podrtía cambiar el 60 por 10 (supuesto se vacnun más rapidin)
                                                      newborn['fecha_nac'] + pd.to_timedelta(np.random.uniform(0, 1, size=len(newborn)), unit='d')) #podría ser entre 0 y 2
    
    newborn.loc[(newborn.rand_cobertura_nirse>cobertura_nirse['newborn']) & ~(newborn['fecha_nac'] < temporada_inicio), 'fecha_colocacion_nirse'] = pd.NaT
    newborn.loc[(newborn.rand_cobertura_nirse>cobertura_nirse['catchup']) & (newborn['fecha_nac'] < temporada_inicio), 'fecha_colocacion_nirse'] = pd.NaT
    newborn.loc[newborn.fecha_colocacion_nirse.notna(),'nirse'] = 1

    newborn['paso_net_inmune'] = ((newborn['fecha_colocacion_nirse'] + pd.Timedelta(days=7)) < newborn['fechaIng']).astype(int)
    
    newborn_inm = newborn[newborn['paso_net_inmune'] == 1]
    newborn_no_inm = newborn[newborn['paso_net_inmune'] == 0]
    
    count_total_seasonal = newborn.nirse.sum() 
    count_no_time_incubar_nirse_seasonal = newborn_no_inm.query('nirse==1').RUN.nunique()
    
    newborn_inm['meses_desde_colocacion'] = (
        (newborn_inm['fechaIng'] - newborn_inm['fecha_colocacion_nirse'])
        / pd.Timedelta(days=30)
    )
    newborn_inm['efectividad_actual'] = np.where(
        newborn_inm['meses_desde_colocacion'] <= duracion_nirse_meses,
        efectividad_nirse_alta,
        efectividad_nirse_alta / 2
    )
    newborn_inm['efectividad_actual_upc'] = np.where(
        newborn_inm['meses_desde_colocacion'] <= duracion_nirse_meses,
        efectividad_nirse_alta_upc,
        efectividad_nirse_alta_upc / 2
    )
    newborn_inm['efectividad_actual_ma'] = np.where(
        newborn_inm['meses_desde_colocacion'] <= duracion_nirse_meses,
        efectividad_nirse_ma,
        efectividad_nirse_ma / 2
    )
    newborn_inm['estado_actual'] = np.where(
        newborn_inm['days_upc'] > 0, 'ICU', 'HA'
    )
    newborn_inm['estado_salida_nirse'] = newborn_inm.apply(
        lambda r: estado_salida_nirse_funct(
            estado_actual=r['estado_actual'],
            eff_MA=r['efectividad_actual_ma'],
            eff_HA=r['efectividad_actual'],
            eff_ICU=r['efectividad_actual_upc']
        ),
        axis=1
    )
    conds = [
        newborn_inm['estado_salida_nirse'] == 'libre',
        newborn_inm['estado_salida_nirse'] == 'MA',
        newborn_inm['estado_salida_nirse'] == 'HA',
        newborn_inm['estado_salida_nirse'] == 'ICU'
    ]
    
    
    newborn_inm['days_mb_plus_upc_mapped'] = ((newborn_inm['days_mb']+newborn_inm['days_upc'])*reduccion_estada_nirse).astype(int)
    newborn_inm['days_mb_only_mapped'] = (newborn_inm['days_mb']*reduccion_estada_nirse).astype(int)
    newborn_inm['days_upc_only_mapped'] = (newborn_inm['days_upc']*reduccion_estada_nirse).astype(int)
    
    choices_mb = [0,0,newborn_inm['days_mb_plus_upc_mapped'],newborn_inm['days_mb_only_mapped']]
    
    choices_upc = [0,0,0,newborn_inm['days_upc_only_mapped']]

    newborn_inm['days_mb_simuled'] = np.select(conds, choices_mb, default=newborn_inm['days_mb'])
    newborn_inm['days_upc_simuled'] = np.select(conds, choices_upc, default=newborn_inm['days_upc'])
    
    newborn_inm['visita_urgencias'] = (newborn_inm['estado_salida_nirse'] == 'MA').astype(int)
    
    newborn_inm.drop(columns=['days_mb_plus_upc_mapped','days_mb_only_mapped','days_upc_only_mapped'],inplace=True) 
    
    newborn = pd.concat([newborn_inm, newborn_no_inm], ignore_index=True)

    df = pd.concat([catchup, newborn], ignore_index=True)
    n_bin = df_nac_tomerge[~df_nac_tomerge['RUN'].isin(df.RUN.unique())].query('@fecha_catchup <= fecha_nac <= @temporada_fin').RUN.nunique() #nacidos no enfermos
    n_nirse = df['nirse'].sum() + np.random.binomial(n_bin,(cobertura_nirse['catchup'] + cobertura_nirse['newborn'])/2)
    
    hosp_upc_dia_base = df.days_upc.sum()
    hosp_simuladas_upc = df.days_upc_simuled.sum()
    hospitalizaciones_ev_upc = hosp_upc_dia_base - hosp_simuladas_upc
    porcent_evit_upc = ((hospitalizaciones_ev_upc)/hosp_upc_dia_base)*100
    costo_hosp_upc_total = hosp_simuladas_upc * costo_upc_per_day
    ahorro_upc = hospitalizaciones_ev_upc * costo_upc_per_day
    
    hosp_basic_dia_base = df.days_mb.sum()
    hosp_simuladas_basic = df.days_mb_simuled.sum()
    hospitalizaciones_ev_basic = hosp_basic_dia_base - hosp_simuladas_basic
    porcent_evit_basic = ((hospitalizaciones_ev_basic)/hosp_basic_dia_base)*100
    costo_hosp_basic_total = hosp_simuladas_basic * costo_hosp_per_day
    ahorro_basic = hospitalizaciones_ev_basic * costo_hosp_per_day
    
    hospitalizaciones_base = hosp_basic_dia_base + hosp_upc_dia_base
    hospitalizaciones_simuladas = hosp_simuladas_basic + hosp_simuladas_upc
    hospitalizaciones_ev = hospitalizaciones_base - hospitalizaciones_simuladas
    costo_hosp_total = costo_hosp_basic_total + costo_hosp_upc_total
    ahorro_hosp= ahorro_basic + ahorro_upc
    
    costo_vacunacion_total = (n_nirse * costo_nirse_total)
    #ahorro_hosp = hospitalizaciones_ev * costo_hosp
    costo_total = costo_vacunacion_total + costo_hosp_total 
    ahorro_total = ahorro_hosp - costo_vacunacion_total 
    
    ######### URGENCIAS ############

    fecha_min_menores_1a = temporada_inicio - pd.Timedelta(days=365)
    
    df_nac_tomerge = (pf_nac[['RUN','SEMANAS','fecha_nac']]
                    .assign(fecha_nac = lambda x: pd.to_datetime(x.fecha_nac,infer_datetime_format=True),
                            year_nac = lambda x: x.fecha_nac.dt.year,
                            month_nac = lambda x: x.fecha_nac.dt.month,
                            week_nac = lambda x: x.fecha_nac.dt.isocalendar().week,)
                    .query('(@fecha_min_menores_1a<=fecha_nac<=@temporada_fin)')
                    .drop_duplicates(subset=['RUN'])
                    .copy())

    df_nac_tomerge.loc[df_nac_tomerge['fecha_nac'] == pd.Timestamp('2023-01-01'),'week_nac'] = 1 ##podría tambien decir que es 2022 y semana 52

    if fecha_catchup != fecha_min_menores_1a:
        nacimientos_normal = (
            df_nac_tomerge
            .assign(group=lambda x: np.where((fecha_catchup <= x.fecha_nac) & (x.fecha_nac < temporada_inicio), 'catchup',
                                            np.where((temporada_inicio <= x.fecha_nac) & (x.fecha_nac <= temporada_fin),'seasonal',
                                                    'despreciable'))
            )      
            .groupby(['group', 'week_nac','year_nac'])
            .size()
            .unstack(0)
            .fillna(0)
            .reset_index()
            .sort_values(by=['year_nac','week_nac'])
        )
        for c in ['catchup','seasonal']:
            nacimientos_normal[f'{c}_cumsum'] = nacimientos_normal[c].cumsum()
        

        nacimientos_normal['despreciable_cumsum'] = (nacimientos_normal['despreciable'].rolling(window=52, min_periods=1).sum()
        )

        nacimientos_normal = (nacimientos_normal
                            .assign(total= lambda x: x.catchup_cumsum + x.seasonal_cumsum + x.despreciable_cumsum,
                                prop_catchup = lambda x: x.catchup_cumsum/x.total,
                                prop_seasonal = lambda x: x.seasonal_cumsum/x.total)
                            .drop(columns=['total'])
                            .rename(columns={'week_nac':'week'})
        )
    else: 
        nacimientos_normal = (
            df_nac_tomerge
            .assign(group=lambda x: np.where((fecha_catchup <= x.fecha_nac) & (x.fecha_nac < temporada_inicio), 'catchup',
                                            np.where((temporada_inicio <= x.fecha_nac) & (x.fecha_nac <= temporada_fin),'seasonal',
                                                    'despreciable'))
            )      
            .groupby(['group', 'week_nac','year_nac'])
            .size()
            .unstack(0)
            .fillna(0)
            .reset_index()
            .sort_values(by=['year_nac','week_nac'])
            .assign(despreciable = 0)
        )
        for c in ['catchup','seasonal']:
            nacimientos_normal[f'{c}_cumsum'] = nacimientos_normal[c].cumsum()
        

        nacimientos_normal['despreciable_cumsum'] = (nacimientos_normal['despreciable'].rolling(window=52, min_periods=1).sum()
        )

        nacimientos_normal = (nacimientos_normal
                            .assign(total= lambda x: x.catchup_cumsum + x.seasonal_cumsum + x.despreciable_cumsum,
                                prop_catchup = lambda x: x.catchup_cumsum/x.total,
                                prop_seasonal = lambda x: x.seasonal_cumsum/x.total)
                            .drop(columns=['total'])
                            .rename(columns={'week_nac':'week'})
        )
        


    df_nac_ola = nacimientos_normal.query('(@temporada_inicio.week<=week<=@temporada_fin.week) & (year_nac==2023)')
    
    df_final = (pd.read_csv(path_data/'atenciones_pub_y_priv_gonzalo_y_amal.csv')
                .query('year==2023')
                .query('week>=@temporada_inicio.week')
                .merge(df_nac_ola[['week','prop_catchup','prop_seasonal']], how='left',on='week')
                .assign(vrs_isp_a_catchup = lambda x: (x.vrs_pub_priv_a * x.prop_catchup).astype(int),
                        vrs_isp_a_seasonal = lambda x: (x.vrs_pub_priv_a * x.prop_seasonal).astype(int),
                        vrs_isp_count_real = lambda x: x.vrs_isp_a_catchup + x.vrs_isp_a_seasonal)
    )
    
        #count_no_time_incubar_nirse/count_total es la proba de quee falle nirse por cubrir dsps de ingreso en catchup
    p_1 = count_no_time_incubar_nirse/count_total
    p_2 = count_no_time_incubar_nirse_seasonal/count_total_seasonal
    
    q_1 = count_total/(count_total_seasonal+ count_total)
    q_2 = count_total_seasonal/(count_total_seasonal+ count_total)
    
    MA_base_catchup = df_final.vrs_isp_a_catchup.sum()
    MA_base_newborn = df_final.vrs_isp_a_seasonal.sum()
    MA_base_total = df_final.vrs_isp_count_real.sum()
    
    
    proba_pasar_net = ((1-p_1)*q_1)+ ((1-p_2)*q_2)
    cober_nirse = ((MA_base_catchup*cobertura_nirse['catchup']) + ((MA_base_newborn)*(cobertura_nirse['newborn'])))*proba_pasar_net
    
    MA_simuled_ev_nirse = np.random.binomial(cober_nirse, efectividad_nirse_ma) # segun paper es 76,4 we did not compute this
    
    MA_simuled_ev = MA_simuled_ev_nirse
    MA_simuled = MA_base_total - MA_simuled_ev + df.visita_urgencias.sum()
    costo_hosp_MA = MA_simuled * costo_emerg
    ahorro_MA = (MA_base_total * costo_emerg) - costo_hosp_MA # = (MA_simuled_ev-df.visita_urgencias.sum()) * costo_emerg
    porcent_evit_MA = (MA_base_total - MA_simuled)/MA_base_total*100 # = (MA_simuled_ev-df.visita_urgencias.sum())/MA_base*100
    
    palivizumab_23 = 12568198
    ahorro_hosp = ahorro_hosp + ahorro_MA
    # ahorro_total = ahorro_hosp - costo_vacunacion_total 
    ahorro_total = ahorro_total + ahorro_MA + palivizumab_23 #+ 13_000_000
    costo_total = costo_total + costo_hosp_MA
    costo_hosp_total = costo_hosp_total + costo_hosp_MA

    #########################################################

    resultados = {
        'costo ICU': round(ahorro_upc/ 1_000, 2),
        'costo_HA': round(ahorro_basic/ 1_000, 2),
        'costo_MA': round(ahorro_MA/ 1_000, 2),
        'costo_total_vacunacion (M)': -round(costo_vacunacion_total / 1_000, 2),
        'costo_pali_savings (M)': round((palivizumab_23) / 1_000, 2),
        'net_direct_cost (M)': round((ahorro_total) / 1_000, 2),
        'porcent_evit_upc': porcent_evit_upc,
        'porcent_evit_basic':porcent_evit_basic,
        'porcent_evit_MA': porcent_evit_MA,

    }
    # print(ahorro_upc + ahorro_basic + ahorro_MA - costo_vacunacion_total + palivizumab_23, 'versus', ahorro_total) son lo mismo
    return resultados, df

resultados , df_simulated = cost_eff_strategy_optimize(df=df, 
                                              duracion_nirse_meses=6, 
                                              fecha_catchup=pd.Timestamp("2022-10-01"), 
                                              temporada_inicio=pd.Timestamp("2023-04-01"))
resultados

{'costo ICU': 12303.96,
 'costo_HA': 7966.6,
 'costo_MA': 6565.63,
 'costo_total_vacunacion (M)': -51574.96,
 'costo_pali_savings (M)': 12568.2,
 'net_direct_cost (M)': -12170.57,
 'porcent_evit_upc': 77.61306189370133,
 'porcent_evit_basic': 69.53960682278115,
 'porcent_evit_MA': 65.08812668925349}

In [ ]:
##################################### ESTO ES CLESRO ###################################
eff_lowers_nirse = [0.433, 0.625, 0.629]
eff_means_nirse = [0.595, 0.813, 0.917]
eff_uppers_nirse = [0.711, 0.907, 0.981]


def loss_dirichlet(alpha, lowers, means, uppers, n_samples=10000, seed=42):
    np.random.seed(seed)
    alpha = np.clip(alpha, 1e-3, None)
    samples = np.random.dirichlet(alpha, size=n_samples)
    
    eff_ma = samples[:, 0]
    eff_ha = samples[:, 0] + samples[:, 1]
    eff_icu = samples[:, 0] + samples[:, 1] + samples[:, 2]

    sim = np.percentile(np.column_stack([eff_ma, eff_ha, eff_icu]), [2.5, 50, 97.5], axis=0)
    
    target = np.array([lowers, means, uppers])
    return np.sum((sim - target)**2)

best_loss = np.inf
best_alpha = None

for trial in range(100):
    x0 = np.random.uniform(0.5, 10, size=4)  # Alpha inicial aleatoria
    res = minimize(loss_dirichlet,
                   x0=x0,
                   args=(eff_lowers_nirse, eff_means_nirse, eff_uppers_nirse),
                   bounds=[(1e-2, 1e3)]*4,
                   method='L-BFGS-B')
    if res.fun < best_loss:
        best_loss = res.fun
        best_alpha = res.x

print("Mejores alphas encontrados:", best_alpha)
print("Pérdida mínima:", best_loss)

Parámetros α óptimos: [10.06324973  3.51528343  2.60462045  0.82958028]
Mejores alphas encontrados: [15.87151326  5.64468931  1.58466766  4.1394156 ]
Pérdida mínima: 0.011492372441445327


In [ ]:
# alpha_diri = [15.87151326, 5.64468931, 1.58466766, 4.1394156 ]

In [ ]:
from itertools import product
import numpy as np
import pandas as pd

def sample_effectivities_from_dirichlet(alpha, n_samples):
    e = np.random.dirichlet(alpha, n_samples)
    return [(v[0], v[0] + v[1], v[0] + v[1] + v[2]) for v in e]

alpha_nirse = [15.87151326, 5.64468931, 1.58466766, 4.1394156] #fit_dirichlet_from_means(eff_means_nirse)

df_nac_tomerge = (pd.read_csv(path_data/'df_nac_tomerge_to_simulate.csv')
                  .query('(year_nac == 2023 & month_nac <= 9) | (year_nac == 2022 & month_nac >=1)')
                  .assign(fecha_nac = lambda x: pd.to_datetime(x.fecha_nac, infer_datetime_format=True),))

df = (pd.read_csv(path_data/'df_historic_process_to_simulate.csv')
      .query('(year_nac == 2023 & month_nac <= 9) | (year_nac == 2022 & month_nac >=1)')
      .assign(fecha_nac = lambda x: pd.to_datetime(x.FECHA_NAC, infer_datetime_format=True),
              fechaIng = lambda x: pd.to_datetime(x.fechaIng, infer_datetime_format=True),
              fechaEgr = lambda x: pd.to_datetime(x.fechaEgr, format='%Y-%m-%d'),
              to_orden = lambda x: (x.fechaEgr - x.fechaIng).dt.days,)
)

# Parámetros a combinar
fechas_catchup = pd.to_datetime(["2022-10-01"])
inicios_temporada = pd.to_datetime(["2023-04-01"])
coberturas_abrysvo = [0.0]

N = 1000
resultados_estrategias = []

for fc, ti, cov_ab in product(fechas_catchup, inicios_temporada, coberturas_abrysvo):
    
    # Samplear N efectividades desde Dirichlet
    effs_nirse = sample_effectivities_from_dirichlet(alpha_nirse, N)
    
    # Guardar métricas
    net_costs = []
    costs_vac = []
    ahorro_ma = []
    ahorro_ha = []
    ahorro_icu = []
    
    for i in range(N):
        eff_ma_nirse, eff_ha_nirse, eff_icu_nirse = effs_nirse[i]
        
        result, _ = cost_eff_strategy_optimize(
            df=df.copy(),
            fecha_catchup=fc,
            temporada_inicio=ti,
            duracion_nirse_meses=6,
            efectividad_nirse_ma = eff_ma_nirse,
            efectividad_nirse_ha = eff_ha_nirse,
            efectividad_nirse_icu = eff_icu_nirse,
        )
        
        # Acumular variables clave
        net_costs.append(result['net_direct_cost (M)'])
        costs_vac.append(result['costo_total_vacunacion (M)'])
        ahorro_ma.append(result['costo_MA'])
        ahorro_ha.append(result['costo_HA'])
        ahorro_icu.append(result['costo ICU'])
    
    # Guardar resumen con ICs
    resumen = {
        'fecha_catchup': fc.strftime('%Y-%m-%d'),
        'temporada_inicio': ti.strftime('%Y-%m-%d'),
        'cobertura_abrysvo': cov_ab,
        'net_direct_cost_mean': np.mean(net_costs),
        'net_direct_cost_low': np.percentile(net_costs, 2.5),
        'net_direct_cost_high': np.percentile(net_costs, 97.5),
        'costo_vac_mean': np.mean(costs_vac),
        'costo_MA_mean': np.mean(ahorro_ma),
        'costo_HA_mean': np.mean(ahorro_ha),
        'costo_ICU_mean': np.mean(ahorro_icu),
        'costo_MA_low': np.percentile(ahorro_ma, 2.5),
        'costo_HA_low': np.percentile(ahorro_ha, 2.5),
        'costo_ICU_low': np.percentile(ahorro_icu, 2.5),
        'costo_MA_high': np.percentile(ahorro_ma, 97.5),
        'costo_HA_high': np.percentile(ahorro_ha, 97.5),
        'costo_ICU_high': np.percentile(ahorro_icu, 97.5),
    }
    
    resultados_estrategias.append(resumen)

tabla_resultados = pd.DataFrame(resultados_estrategias)
tabla_resultados[['fecha_catchup','temporada_inicio','cobertura_abrysvo','net_direct_cost_mean','costo_vac_mean',
 'costo_MA_mean','costo_HA_mean','costo_ICU_mean']]


In [ ]:
tabla_resultados[['fecha_catchup','temporada_inicio','cobertura_abrysvo','net_direct_cost_mean','costo_vac_mean',
 'costo_MA_mean','costo_HA_mean','costo_ICU_mean']].to_excel(path_data/'monte_carlo_1_select.xlsx',index=False)

tabla_resultados.to_excel(path_data/'monte_carlo_1.xlsx',index=False)